# J3 — Analytics SQL & Couche Gold (Olist)

Notebook autonome à exécuter **après J2**. Il lit `/lakehouse/silver/orders_enriched`, crée quatre tables Gold Delta et contrôle leurs résultats.

**Pré-requis :** NameNode/HDFS opérationnel, table Silver déjà écrite. Après une erreur `ConnectionRefusedError`, redémarrer le kernel Jupyter avant de lancer ce notebook. Ne pas lancer J2 en parallèle pour limiter les conflits de mémoire/ports.

## 1 — Spark et chemins HDFS

In [6]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType

# Utilise la configuration Spark/Delta déjà fournie par le conteneur Jupyter.
# Ne pas appeler spark.stop() dans ce notebook.
spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")

HDFS = "hdfs://namenode:9000"
SILVER_MAIN_PATH = f"{HDFS}/lakehouse/silver/orders_enriched"
GOLD_ROOT = f"{HDFS}/lakehouse/gold"

print("Spark :", spark.version)
print("Spark UI :", spark.sparkContext.uiWebUrl)
print("Test Spark :", spark.range(5).count())

Spark : 4.2.0
Spark UI : http://spark-jupyter:4042
Test Spark : 5


## 2 — Lecture Silver et vérifications

In [7]:
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
silver.createOrReplaceTempView("v_silver")

required = {
    "order_id", "customer_unique_id", "customer_state", "product_category",
    "order_value", "item_count", "delivery_days", "delay_days", "is_late",
    "purchase_month", "order_status"
}
missing = sorted(required - set(silver.columns))
if missing:
    raise ValueError(f"Colonnes Silver manquantes : {missing}. Vérifier J2.")

silver_count = silver.count()
unique_orders = silver.select("order_id").distinct().count()
print(f"Silver : {silver_count:,} commandes | IDs distincts : {unique_orders:,}")
assert silver_count == unique_orders, "La table Silver contient des order_id dupliqués."
if silver_count != 99441:
    print("ATTENTION : nombre de commandes différent des 99 441 attendues pour Olist.")
silver.select("order_id", "purchase_month", "customer_state", "order_value", "review_score").show(5, truncate=False)

Silver : 99,441 commandes | IDs distincts : 99,441


+--------------------------------+--------------+--------------+-----------+------------+
|order_id                        |purchase_month|customer_state|order_value|review_score|
+--------------------------------+--------------+--------------+-----------+------------+
|00054e8431b9d7675808bcb819fb4a32|2017-12       |SP            |31.75      |4.0         |
|000576fe39319847cbb9d288c5617fa6|2018-07       |SP            |880.75     |5.0         |
|00061f2a7bc09da83e415a52dc8a4af1|2018-03       |SP            |68.87      |5.0         |
|000e562887b1f2006d75e0be9558292e|2018-02       |SP            |41.11      |3.0         |
|000e906b789b55f64edcb1f84030f90d|2017-11       |SP            |33.84      |3.0         |
+--------------------------------+--------------+--------------+-----------+------------+
only showing top 5 rows


## 3 — Préparation des colonnes analytiques

Les commandes peuvent avoir plusieurs catégories de produits. La table Silver J2 conserve une catégorie représentative par commande : le chiffre d’affaires par catégorie ci-dessous est donc une **attribution par commande**, pas une ventilation exacte au niveau de chaque article.

In [8]:
base = (
    silver
    .withColumn("order_value_num", F.expr("try_cast(order_value AS DOUBLE)"))
    .withColumn("delivery_days_num", F.expr("try_cast(delivery_days AS DOUBLE)"))
    .withColumn("delay_days_num", F.expr("try_cast(delay_days AS DOUBLE)"))
    .withColumn("is_late_num", F.expr("try_cast(is_late AS INT)"))
    .withColumn("review_score_num", F.expr("try_cast(review_score AS DOUBLE)"))
    .withColumn("product_category_clean", F.coalesce(F.col("product_category"), F.lit("unknown")))
    .withColumn("customer_state_clean", F.coalesce(F.col("customer_state"), F.lit("unknown")))
)
base.createOrReplaceTempView("v_analytics")
print("Vue v_analytics disponible :", base.count(), "lignes")

[Stage 104:=====================================================> (49 + 1) / 50]

Vue v_analytics disponible : 99441 lignes


## 4 — Gold : performance par catégorie

In [9]:
category_performance = spark.sql("""
SELECT
    product_category_clean AS product_category,
    COUNT(*) AS total_orders,
    ROUND(SUM(COALESCE(order_value_num, 0)), 2) AS total_revenue,
    ROUND(AVG(order_value_num), 2) AS avg_order_value,
    ROUND(AVG(review_score_num), 2) AS avg_review_score,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days
FROM v_analytics
GROUP BY product_category_clean
""")
category_performance.orderBy(F.desc("total_revenue")).show(15, truncate=False)

+----------------------+------------+-------------+---------------+----------------+-----------------+
|product_category      |total_orders|total_revenue|avg_order_value|avg_review_score|avg_delivery_days|
+----------------------+------------+-------------+---------------+----------------+-----------------+
|beleza_saude          |8796        |1440061.38   |163.72         |4.19            |12.1             |
|relogios_presentes    |5601        |1303159.52   |232.67         |4.07            |12.82            |
|cama_mesa_banho       |9315        |1241570.64   |133.29         |3.98            |13.03            |
|esporte_lazer         |7683        |1158184.46   |150.75         |4.17            |12.26            |
|informatica_acessorios|6661        |1059496.8    |159.06         |4.03            |13.19            |
|moveis_decoracao      |6355        |906602.02    |142.66         |4.02            |13.11            |
|utilidades_domesticas |5836        |781450.26    |133.9          |4.15  

## 5 — Gold : retards par État

Le taux de retard est calculé **uniquement sur les commandes dont la date réelle de livraison est disponible**. Les commandes non livrées ne sont pas considérées comme ponctuelles.

In [10]:
late_orders_by_state = spark.sql("""
SELECT
    customer_state_clean AS customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS late_rate_pct,
    ROUND(AVG(delay_days_num), 2) AS avg_delay_days,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days
FROM v_analytics
GROUP BY customer_state_clean
""")
late_orders_by_state.orderBy(F.desc("late_rate_pct")).show(30, truncate=False)

+--------------+------------+----------------+-----------+-------------+--------------+-----------------+
|customer_state|total_orders|delivered_orders|late_orders|late_rate_pct|avg_delay_days|avg_delivery_days|
+--------------+------------+----------------+-----------+-------------+--------------+-----------------+
|AL            |413         |397             |95         |23.93        |-8.03         |24.54            |
|MA            |747         |717             |141        |19.67        |-8.89         |21.57            |
|PI            |495         |476             |76         |15.97        |-10.63        |19.46            |
|CE            |1336        |1279            |196        |15.32        |-10.11        |21.27            |
|SE            |350         |335             |51         |15.22        |-9.33         |21.52            |
|BA            |3380        |3256            |457        |14.04        |-10.1         |19.34            |
|RJ            |12852       |12353           |

## 6 — Gold : évolution mensuelle des livraisons

In [11]:
monthly_delivery = spark.sql("""
SELECT
    purchase_month,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delivery_days_num IS NOT NULL THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS late_rate_pct,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days,
    ROUND(SUM(COALESCE(order_value_num, 0)), 2) AS total_revenue
FROM v_analytics
WHERE purchase_month IS NOT NULL
GROUP BY purchase_month
""")
monthly_delivery.orderBy("purchase_month").show(40, truncate=False)

+--------------+------------+----------------+-----------+-------------+-----------------+-------------+
|purchase_month|total_orders|delivered_orders|late_orders|late_rate_pct|avg_delivery_days|total_revenue|
+--------------+------------+----------------+-----------+-------------+-----------------+-------------+
|2016-09       |4           |1               |1          |100.00       |54.81            |387.21       |
|2016-10       |324         |270             |3          |1.11         |19.58            |59090.48     |
|2016-12       |1           |1               |0          |0.00         |4.69             |19.62        |
|2017-01       |800         |750             |23         |3.07         |12.65            |138488.04    |
|2017-02       |1780        |1653            |53         |3.21         |13.17            |291908.01    |
|2017-03       |2682        |2546            |142        |5.58         |12.95            |449863.6     |
|2017-04       |2404        |2303            |181      

## 7 — Gold : segmentation des clients (RFM simplifiée)

Les segments sont fondés sur le nombre de commandes et la dépense totale par client, sans seuil arbitraire de récence. La table conserve une ligne par `customer_unique_id`.

In [12]:
customer_stats = (
    base.filter(F.col("customer_unique_id").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum(F.coalesce("order_value_num", F.lit(0.0))), 2).alias("total_spent"),
        F.max("purchase_date").alias("last_purchase_date")
    )
)
customer_segments = (
    customer_stats
    .withColumn(
        "segment",
        F.when(F.col("total_orders") >= 3, "loyal_3plus")
         .when(F.col("total_orders") == 2, "repeat_2")
         .otherwise("one_time")
    )
)
customer_segments.groupBy("segment").agg(
    F.count("*").alias("customers"),
    F.round(F.sum("total_spent"), 2).alias("revenue")
).orderBy("segment").show(truncate=False)

[Stage 127:>                                                        (0 + 4) / 4]

+-----------+---------+-------------+
|segment    |customers|revenue      |
+-----------+---------+-------------+
|loyal_3plus|252      |134667.32    |
|one_time   |93099    |1.506498438E7|
|repeat_2   |2745     |809355.39    |
+-----------+---------+-------------+



## 8 — Contrôles avant écriture Gold

In [13]:
gold_tables = {
    "category_performance": category_performance,
    "late_orders_by_state": late_orders_by_state,
    "monthly_delivery": monthly_delivery,
    "customer_segments": customer_segments,
}
for name, df in gold_tables.items():
    rows = df.count()
    print(f"[CHECK] {name}: {rows:,} lignes")
    assert rows > 0, f"Table Gold vide : {name}"
assert customer_segments.count() == customer_segments.select("customer_unique_id").distinct().count()
assert category_performance.agg(F.sum("total_orders")).first()[0] == silver_count
assert late_orders_by_state.agg(F.sum("total_orders")).first()[0] == silver_count
print("Contrôles Gold validés.")

[CHECK] category_performance: 74 lignes


[CHECK] late_orders_by_state: 27 lignes


[CHECK] monthly_delivery: 25 lignes


[CHECK] customer_segments: 96,096 lignes


Contrôles Gold validés.


## 9 — Écriture Delta des quatre tables Gold

L’écriture utilise `overwrite` sur les **seuls chemins Gold ciblés**. Les données Bronze et Silver ne sont pas modifiées.

In [14]:
for name, df in gold_tables.items():
    path = f"{GOLD_ROOT}/{name}"
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    print(f"[GOLD OK] {path}")

[GOLD OK] hdfs://namenode:9000/lakehouse/gold/category_performance


[GOLD OK] hdfs://namenode:9000/lakehouse/gold/late_orders_by_state


[GOLD OK] hdfs://namenode:9000/lakehouse/gold/monthly_delivery


[Stage 245:===================================================>   (47 + 3) / 50]

[GOLD OK] hdfs://namenode:9000/lakehouse/gold/customer_segments


## 10 — Relecture, validation et requêtes SQL

In [15]:
for name in gold_tables:
    path = f"{GOLD_ROOT}/{name}"
    saved = spark.read.format("delta").load(path)
    print(f"[DELTA RELU] {name}: {saved.count():,} lignes")
    saved.createOrReplaceTempView(f"gold_{name}")

spark.sql("""
SELECT product_category, total_orders, total_revenue
FROM gold_category_performance
ORDER BY total_revenue DESC
LIMIT 10
""").show(truncate=False)

print("J3 terminé : quatre tables Gold Delta disponibles.")

[DELTA RELU] category_performance: 74 lignes


[DELTA RELU] late_orders_by_state: 27 lignes


[DELTA RELU] monthly_delivery: 25 lignes


[DELTA RELU] customer_segments: 96,096 lignes


+----------------------+------------+-------------+
|product_category      |total_orders|total_revenue|
+----------------------+------------+-------------+
|beleza_saude          |8796        |1440061.38   |
|relogios_presentes    |5601        |1303159.52   |
|cama_mesa_banho       |9315        |1241570.64   |
|esporte_lazer         |7683        |1158184.46   |
|informatica_acessorios|6661        |1059496.8    |
|moveis_decoracao      |6355        |906602.02    |
|utilidades_domesticas |5836        |781450.26    |
|cool_stuff            |3604        |720981.29    |
|automotivo            |3881        |685479.05    |
|ferramentas_jardim    |3482        |584380.24    |
+----------------------+------------+-------------+

J3 terminé : quatre tables Gold Delta disponibles.


## 11 — Historique Delta et contrôle final (facultatif)

In [16]:
from delta.tables import DeltaTable
for name in gold_tables:
    path = f"{GOLD_ROOT}/{name}"
    print("Historique :", name)
    DeltaTable.forPath(spark, path).history(3).select("version", "timestamp", "operation").show(truncate=False)

print("J3 prêt pour J4 / dashboard Streamlit.")

Historique : category_performance


+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-11-21 14:50:55.649|WRITE    |
|0      |2026-11-21 14:41:19.774|WRITE    |
+-------+-----------------------+---------+

Historique : late_orders_by_state


+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-11-21 14:51:04.875|WRITE    |
|0      |2026-11-21 14:40:37.726|WRITE    |
+-------+-----------------------+---------+

Historique : monthly_delivery
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-11-21 14:51:15.585|WRITE    |
|0      |2026-11-21 14:40:59.096|WRITE    |
+-------+-----------------------+---------+

Historique : customer_segments
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-11-21 14:51:23.619|WRITE    |
+-------+-----------------------+---------+

J3 prêt pour J4 / dashboard Streamlit.
